In [3]:
import pandas as pd
import numpy as np

#Decision Tree class
class DecisionTree:
    #Initializes tree class
    def __init__(self, dataset, min_node_size, min_purity):
        self.dataset = dataset
        self.min_node_size = min_node_size
        self.min_purity = min_purity
        self.root = self.build_tree(dataset)

    #Finds purity of the dataset
    def purity(self, dataset):
        class_counts = dataset['quality'].value_counts()
        majority_class = class_counts.idxmax()
        return class_counts[majority_class] / len(dataset)

    #Implentation of Evalute Numeric Attribute (algorithm 19.2)
    def evaluate_numeric_attribute(self, dataset, attribute):
        #Information Gain

        #Find entropy of data set
        def entropy(data):
            class_counts = data['quality'].value_counts()
            total_samples = len(data)
            entropy = 0
            for class_label, count in class_counts.items():
                p = count / total_samples
                entropy -= p * np.log2(p)

            return entropy

        #Calculate conditional entropy post split
        def conditional_entropy(data, split_value, attribute):
            data_left = data[data[attribute] <= split_value]
            data_right = data[data[attribute] > split_value]
            total_samples = len(data)
            p_left = len(data_left) / total_samples
            p_right = len(data_right) / total_samples
            entropy_left = entropy(data_left)
            entropy_right = entropy(data_right)
            conditional_entropy = p_left * entropy_left + p_right * entropy_right
            return conditional_entropy

        dataset = dataset.sort_values(attribute)
        split_values = dataset[attribute].unique()
        best_split = None
        best_info_gain = 0
        for i in range(1, len(split_values)):
            split_value = (split_values[i - 1] + split_values[i]) / 2
            info_gain = entropy(dataset) - conditional_entropy(dataset, split_value, attribute)

            if info_gain > best_info_gain:
                best_info_gain = info_gain
                best_split = split_value
        return best_split, best_info_gain


    #Find best split point from numeric attributes
    def find_best_split(self, dataset):
        best_split = None
        best_score = 0
        for attribute in dataset.columns[:-1]:
            if dataset[attribute].dtype == np.float64:
                v, score = self.evaluate_numeric_attribute(dataset, attribute)
                if score > best_score:
                    best_split = (attribute, v)
                    best_score = score
        return best_split, best_score

    #Builds the tree
    def build_tree(self, dataset):
        n = len(dataset)

        #Stop conditions for tree
        if n <= self.min_node_size or self.purity(dataset) >= self.min_purity:
            majority_class = dataset['quality'].mode().iloc[0]
            return {'leaf': True, 'class': majority_class, 'p': 1.0, 'size': n}

        #Cinds best split point
        best_split, best_score = self.find_best_split(dataset)

        if best_split is None:
            majority_class = dataset['quality'].mode().iloc[0]
            return {'leaf': True, 'class': majority_class, 'p': 1.0, 'size': n}

        #Splits the data
        left_data = dataset[dataset[best_split[0]] <= best_split[1]]
        right_data = dataset[dataset[best_split[0]] > best_split[1]]

        #Recursively calls left and right branches of the tree
        left_subtree = self.build_tree(left_data)
        right_subtree = self.build_tree(right_data)

        return {'attribute': best_split[0], 'split_value': best_split[1],
                'left': left_subtree, 'right': right_subtree}

    #Prints the tree
    def print_tree(self, node, depth=0):
        if 'leaf' in node:
            print(f"{'  ' * depth}| leaf: class={node['class']}, p={node['p']}")
        else:
            print(f"{'  ' * depth}{node['attribute']} <= {node['split_value']}")
            if 'left' in node:
                self.print_tree(node['left'], depth + 1)
            if 'right' in node:
                self.print_tree(node['right'], depth + 1)

    def predict(self, data):
        predictions = []
        
        def traverse(node, sample):
            if 'leaf' in node:
                predictions.append(node['class'])
            else:
                attribute = node['attribute']
                split_value = node['split_value']
                if sample[attribute] <= split_value:
                    traverse(node['left'], sample)
                else:
                    traverse(node['right'], sample)

        for _, sample in data.iterrows():
            traverse(self.root, sample)

        return predictions


#Loads in the dataset
wine_white = pd.read_csv("winequality-white.csv", delimiter=";")
wine_red = pd.read_csv("winequality-red.csv", delimiter=";")

#Creates the tree
min_node_size = 10
min_purity = 0.95
white_tree = DecisionTree(wine_white, min_node_size, min_purity)
red_tree = DecisionTree(wine_red, min_node_size, min_purity)

#Prints red tree
print("Decision Tree for White Wine:")
white_tree.print_tree(white_tree.root)

#Predict white labels
predicted_labels_white = white_tree.predict(wine_white)
actual_labels_white = wine_white['quality']

#Accuracy for white data set
accuracy_white = (predicted_labels_white == actual_labels_white).mean()
print("Accuracy for White Wine Dataset:", accuracy_white)

#Print red tree
print("\nDecision Tree for Red Wine:")
red_tree.print_tree(red_tree.root)

#Predict red labels
predicted_labels_red = red_tree.predict(wine_red)
actual_labels_red = wine_red['quality']

#Accuracy for red data set
accuracy_red = (predicted_labels_red == actual_labels_red).mean()
print("Accuracy for Red Wine Dataset:", accuracy_red)

if accuracy_white > accuracy_red:
    print("Accuracy for White Wine Dataset is bigger")
else:
    print("Accuracy for Red Wine Dataset is bigger")


Decision Tree for White Wine:
alcohol <= 10.850000000000001
  volatile acidity <= 0.2525
    volatile acidity <= 0.2075
      density <= 0.99788
        sulphates <= 0.535
          residual sugar <= 2.1500000000000004
            free sulfur dioxide <= 13.5
              chlorides <= 0.0335
                | leaf: class=5, p=1.0
                | leaf: class=5, p=1.0
              fixed acidity <= 7.95
                density <= 0.99162
                  | leaf: class=6, p=1.0
                  fixed acidity <= 6.35
                    citric acid <= 0.335
                      total sulfur dioxide <= 161.0
                        fixed acidity <= 5.25
                          | leaf: class=6, p=1.0
                          chlorides <= 0.0535
                            | leaf: class=5, p=1.0
                            | leaf: class=5, p=1.0
                        | leaf: class=6, p=1.0
                      pH <= 3.1950000000000003
                        | leaf: class=5, p=1.0
